In [ ]:
import random
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.distributions.categorical import Categorical
from collections import deque

In [917]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [918]:
NROW = 4
NCOL = 4
MAX_T = 2048
BATCH_SIZE = 256

### 2048 Game logic

In [919]:
action_space = ["left", "up", "right", "down"]

In [ ]:
class Game2048:
  """Game logic for the 2048 game."""
  def __init__(self, nrow=NROW, ncol=NCOL):
    self.board = np.zeros((NROW, NCOL), dtype=int)
    self.total_score = 0
    self.log_score = 0
    self.game_is_over = False
    self.generate_new_num()
    self.generate_new_num()
    self.empty_tiles = (NROW * NCOL) - 2


  def check_game_over(self):
    """Check if the game is over given that we have a full board."""
    
    # Check rows
    for row in range(NROW):
      prev_tile = 0
      for col in range(NCOL):
        tile = self.board[row][col]
        if tile == prev_tile:
          return False
        prev_tile = tile

    # Check columns
    a = self.board.transpose()
    for row in range(NROW):
      prev_tile = 0
      for col in range(NCOL):
        tile = a[row][col]
        if tile == prev_tile:
          return False
        prev_tile = tile

    return True

  def random_empty_tile(self):
    """Picks a random empty tile from the board."""

    # Find all empty tiles
    empty_tiles = []
    for i in range(len(self.board)):
      for j in range(len(self.board)):
        if self.board[i][j] == 0:
          empty_tiles.append([i, j])

    self.empty_tiles = len(empty_tiles)

    # If there are no more empty tiles, check if the game is over
    if len(empty_tiles) == 0:
      game_over = self.check_game_over()
      if game_over:
        return None, None
      else:
        return -1, -1

    # Return a random empty tile
    random_row, random_col = random.choice(empty_tiles)
    return random_row, random_col


  def generate_new_num(self):
    """Generate a new random number (2 or 4) and find an empty tile to place it on."""

    # Generate random numbers and empty tile
    new_num = 1 if random.random() < 0.9 else 2
    random_row, random_col = self.random_empty_tile()

    # Stop the game or continue if applicable
    if random_row is None or random_col is None:
        self.game_is_over = True
        return
    elif random_row == -1 or random_col == -1:
      pass
    else:
      self.board[random_row][random_col] = new_num

  def move_left(self):
    """Returns the new board state after a left movement."""

    # Shift all tiles to the left
    for row in range(NROW):
      empty_space = 0
      for col in range(NCOL):
        tile = self.board[row][col]
        if tile == 0:
          empty_space += 1
        elif empty_space > 0:
          self.board[row][col-empty_space] = tile
          self.board[row][col] = 0

      # Add only non-zero values
      new_tiles = []
      for val in self.board[row]:
        if val != 0:
          new_tiles.append(val)

      # Merge the tiles to the left
      prev_tile = 0
      i = 0
      while i < len(new_tiles):
        tile = new_tiles[i]
        if tile == prev_tile:
          merged_tile_val = tile + 1
          new_tiles[i-1] = merged_tile_val
          new_tiles.pop(i)
          prev_tile = 0
          self.total_score += (2 ** tile) * 2
          self.log_score += 2 * tile

        else:
          prev_tile = tile
          i += 1

      # Append empty tiles
      while len(new_tiles) < NCOL:
        new_tiles.append(0)

      # Update row
      self.board[row] = new_tiles

    self.generate_new_num()

  def move_up(self):
    """Returns the new board state after an up movement."""
    self.board = self.board.transpose()
    self.move_left()
    self.board = self.board.transpose()

  def move_right(self):
    """Returns the new board state after a right movement."""
    self.board = self.board[:, ::-1]
    self.move_left()
    self.board = self.board[:, ::-1]


  def move_down(self):
    """Returns the new board state after a down movement."""
    self.board = self.board.transpose()[:, ::-1]
    self.move_left()
    self.board = self.board[:, ::-1].transpose()

  def take_action(self, action):
    """Returns the board state from taking the given action."""
    if action == 0: # left
      self.move_left()
    elif action == 1: # up
      self.move_up()
    elif action == 2: # right
      self.move_right()
    elif action == 3: # down
      self.move_down()



In [921]:
game = Game2048(nrow=NROW, ncol=NCOL)
print(2 ** game.board)

[[1 1 1 1]
 [1 1 2 1]
 [1 1 1 1]
 [2 1 1 1]]


### Define the action space, state space and hyperparameters

In [922]:
# Define the action and state space
a_size = 4 # left, up, right, down arrow keys
s_size = 16 # 4x4 board
h_size = 256 # Neural Network neurons

In [923]:
# PPO Hyperparameters
hp = {
    "clip_range": 0.2,
    "n_training_episodes": 500000,
    "n_evaluation_episodes": 10,
    "gamma": 0.9,
    "gae_lambda": 0.9,
    "lr": 2.5e-4,    
    "value_loss_coefficient": 0.1,
    "entropy_coefficient": 0.01,
}

### Actor Critic Neural Network

In [ ]:
class Agent(nn.Module):
    """
    Actor-Critic Neural Network for PPO training.
    Uses a shared CNN feature extractor to extract features from the board,
    with separate actor and critic heads for policy and value estimation.
    """
    def __init__(self, a_size, h_size=256):
        super().__init__()

        # Shared CNN feature extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=2),
            nn.Tanh(),
            nn.Conv2d(64, 128, kernel_size=2),
            nn.Tanh(),
            nn.Flatten(),
            nn.Linear(h_size * 2, h_size),
            nn.Tanh()
        )

        # Actor head
        self.actor = nn.Linear(h_size, a_size)

        # Critic head
        self.critic = nn.Linear(h_size, 1)

    def get_features(self, x):
        """Extracts shared CNN features from the board."""
        x = self.cnn(x)
        return x

    def get_value(self, x):
        """Returns critic value estimate for a given state."""
        features = self.get_features(x)
        return self.critic(features)

    def get_action_and_value(self, x, action=None):
        """Samples an action and returns log_prob, entropy and critic value estimate"""
        features = self.get_features(x)
        logits = self.actor(features)
        probs = Categorical(logits=logits)
        
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(features)

### PPO Memory

In [ ]:
class PPOMemory:
    """Stores the PPO memory arrays necessary for the algorithm."""
    def __init__(self, discount, gae_lambda, clip_range, value_loss_coef, entropy_coef):
        self.states = np.zeros((MAX_T, 1, 4, 4), dtype=np.float32)
        self.log_probs = np.zeros(MAX_T)
        self.values = np.zeros(MAX_T)
        self.actions = np.zeros(MAX_T)

        self.td_errors = np.zeros(MAX_T)
        self.advantages = np.zeros(MAX_T)
        
        self.batches = []

        self.discount = discount
        self.gae_lambda = gae_lambda
        self.clip_range = clip_range
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef

        self.score = 0
        self.actual_steps = 0

        self.milestones = [7, 8, 9, 10, 11]
        self.reached_milestones = set()

    def update_memory(self, state, log_prob, val, action, new_val, score_diff, reward, i):
        """Updates the PPO arrays with given values."""
        val = val.item()
        log_prob = log_prob.item()
        new_val = new_val.item()

        self.states[i] = state.numpy()
        self.log_probs[i] = log_prob
        self.values[i] = val
        self.actions[i] = action

        self.score += score_diff

        td_error_t = reward + (self.discount * new_val) - val
        self.td_errors[i] = td_error_t

    
    def subset_arrays(self):
        """Subset the arrays for splitting into batches."""
        self.states = self.states[:self.actual_steps]
        self.log_probs = self.log_probs[:self.actual_steps]
        self.values = self.values[:self.actual_steps]
        self.actions = self.actions[:self.actual_steps]
        self.td_errors = self.td_errors[:self.actual_steps]
        self.advantages = np.zeros(self.actual_steps)


    def calculate_advantage(self):
        """Calculate the advantage estimates"""
        last_gae_lam = 0
        for t in range(len(self.td_errors)-1, -1, -1):
            advantage = self.td_errors[t] + (self.discount * self.gae_lambda * (last_gae_lam))
            self.advantages[t] = advantage
            last_gae_lam = advantage
    

    def shuffle_batches(self):
        """Shuffle the arrays into 8 batches."""
        
        # Shuffle the MAX_T steps
        indices = np.arange(self.actual_steps)
        np.random.shuffle(indices)

        # Create 8 batches
        BATCH_SIZE = max(1, math.ceil(self.actual_steps / 8))

        # Place the transitions into batches
        batch_start = np.arange(0, self.actual_steps, BATCH_SIZE)
        self.batches = [indices[i:i+BATCH_SIZE] for i in batch_start]

        # Apply z-score normalization to the advantage
        advantages = (self.advantages - self.advantages.mean()) / (self.advantages.std() + 1e-8)          

        self.advantages = torch.from_numpy(advantages).float()
        self.values = torch.from_numpy(self.values).float()



### Train the PPO agent

In [ ]:
def calculate_score(game, action):
    """Calculate the score gained from taking an action."""

    # Take the action and calculate score difference
    prev_total_score = game.total_score
    prev_log_score = game.log_score
    game.take_action(action)

    score_diff = game.total_score - prev_total_score
    merge_reward = game.log_score - prev_log_score

    return score_diff, merge_reward

In [ ]:
def calculate_reward(ppo_mem, game, merge_reward):
    """Calculate the reward given to the agent for taking a step."""

    # Prepare bonus rewards
    corner_bonus = 0
    milestone_bonus = 0

    flat_board = game.board.flatten()
    max_tile = max(flat_board)
    corners = [flat_board[0], flat_board[3], flat_board[12], flat_board[15]]

    # 1. Penalize each move
    move_penalty = -2

    # 2. Empty tile bonus
    empty_bonus = - (game.empty_tiles * 0.1)

    # 3. Corner bonus
    if max_tile in corners:
        corner_bonus = 1

    # 4. Tile milestone bonus
    if max_tile in ppo_mem.milestones and max_tile not in ppo_mem.reached_milestones:
        ppo_mem.reached_milestones.add(max_tile)
        milestone_bonus = 1

    # Calculate reward for taking step
    reward = merge_reward + move_penalty + empty_bonus + corner_bonus + milestone_bonus 
    return reward

In [ ]:
def step(agent, ppo_mem, game, state, i):
    """Sample an action to take, calculate rewards and update PPO memory."""

    done = False

    # Get action and value
    action, log_prob, entropy, val = agent.get_action_and_value(state)
    action = action.item()

    # Take the action and calc score
    score_diff, merge_reward = calculate_score(game, action)

    # Update state and val
    new_state = torch.from_numpy(game.board).float().unsqueeze(0).unsqueeze(0)
    new_val = agent.get_value(new_state)

    # Calculate reward and add bonuses 
    reward = calculate_reward(ppo_mem, game, merge_reward)

    # Update PPO memory
    ppo_mem.update_memory(state, log_prob, val, action, new_val, score_diff, reward, i)

    # Update the state
    state = new_state
    
    # Stop taking steps if the game is over
    if game.game_is_over is True:
        done = True

    if done or i == MAX_T - 1:
        ppo_mem.actual_steps = i + 1
        return True, state

    return False, state



In [ ]:
def update_weights(ppo_mem, agent, batch, optimizer):
    """Calculate error and update the weights of the agent neural network."""
    
    states_batch = torch.tensor(ppo_mem.states[batch], dtype=torch.float32)
    log_probs_batch = torch.tensor(ppo_mem.log_probs[batch], dtype=torch.float32)
    actions_batch = torch.tensor(ppo_mem.actions[batch], dtype=torch.long)

    # Forward pass of batch data
    new_action, new_log_prob, entropy, critic_val = agent.get_action_and_value(states_batch, action=actions_batch)

    # Weighted clipped probability
    prob_ratio = new_log_prob.exp() / log_probs_batch.exp()

    weighted_probs = ppo_mem.advantages[batch] * prob_ratio
    weighted_clipped_probs = torch.clamp(
        prob_ratio, 1 - ppo_mem.clip_range, 1 + ppo_mem.clip_range
    ) * ppo_mem.advantages[batch]
    
    # Calculate loss
    actor_loss = -torch.min(weighted_probs, weighted_clipped_probs).mean()

    returns = ppo_mem.advantages[batch] + ppo_mem.values[batch]
    critic_loss = (returns - critic_val) ** 2
    critic_loss = critic_loss.mean()

    entropy_loss = entropy.mean()
    total_loss = actor_loss + ppo_mem.value_loss_coef * \
                    critic_loss - ppo_mem.entropy_coef * entropy_loss

    # Backpropagate and update weights
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

In [ ]:
def ppo_algorithm(agent, hp, optimizer):
    """Runs the PPO algorithm to train the agent."""
    clip_range = hp["clip_range"]
    discount = hp["gamma"]
    gae_lambda = hp["gae_lambda"]
    episodes = hp["n_training_episodes"]
    value_loss_coef = hp["value_loss_coefficient"]
    entropy_coef = hp["entropy_coefficient"]

    # Keep track of the last 100 scores
    scores_deque = deque(maxlen=100)
    max_tile_deque = deque(maxlen=100)

    # Create new instance of game and PPO memory for each training episode
    for episode in range(1, episodes+1):
        game = Game2048(NROW, NCOL)
        state = torch.from_numpy(game.board).float().unsqueeze(0).unsqueeze(0)
        ppo_mem = PPOMemory(discount, gae_lambda, clip_range, value_loss_coef, entropy_coef)

        # Take an action in the environment until done or MAX_T reached
        for i in range(MAX_T):
            done, state = step(agent, ppo_mem, game, state, i)
            if done:
                scores_deque.append(ppo_mem.score)
                break

        # Prepare arrays to calcuate error 
        ppo_mem.subset_arrays()
        ppo_mem.calculate_advantage()
        ppo_mem.shuffle_batches()
        max_tile_deque.append(max(game.board.flatten()))

        # Update weights in batches 
        for batch in ppo_mem.batches:
            update_weights(ppo_mem, agent, batch, optimizer)
        
        # Print out scores every 100 games
        if episode % 100 == 0:
            print(f"Episode: {episode}, Average Score (last 100 scores): {np.mean(scores_deque)}, Current Episode Score: {ppo_mem.score}, Max Score: {np.max(scores_deque)}, Max Tile: {2 ** np.max(max_tile_deque)}")
            print("Highest Tile Reached (last 100 scores):")
            print(f"128:  {max_tile_deque.count(7)}%\n256:  {max_tile_deque.count(8)}%\n512:  {max_tile_deque.count(9)}%\n1024: {max_tile_deque.count(10)}%\n2048: {max_tile_deque.count(11)}%")
            print(2 ** game.board)
            print()
            

### Run the PPO algorithm

In [931]:
agent = Agent(a_size, h_size)
optimizer = optim.Adam(agent.parameters(), lr=hp["lr"])

In [932]:
ppo_algorithm(agent, hp, optimizer)

Episode: 100, Average Score (last 100 scores): 931.76, Current Episode Score: 568, Max Score: 2432, Max Tile: 256
Highest Tile Reached (last 100 scores):
128:  40%
256:  3%
512:  0%
1024: 0%
2048: 0%
[[ 2 16  4  2]
 [ 4  8  2  8]
 [ 2 32 64  2]
 [16  4  2  8]]

Episode: 200, Average Score (last 100 scores): 1065.32, Current Episode Score: 1452, Max Score: 3108, Max Tile: 256
Highest Tile Reached (last 100 scores):
128:  48%
256:  7%
512:  0%
1024: 0%
2048: 0%
[[  2  16   4   2]
 [  4  64   2   8]
 [ 16 128  32   2]
 [  2  32  16   4]]

Episode: 300, Average Score (last 100 scores): 958.76, Current Episode Score: 1220, Max Score: 3052, Max Tile: 256
Highest Tile Reached (last 100 scores):
128:  33%
256:  5%
512:  0%
1024: 0%
2048: 0%
[[  2   4  16   2]
 [  4  32 128   4]
 [  8  16  32  16]
 [  4   8  16   4]]

Episode: 400, Average Score (last 100 scores): 951.96, Current Episode Score: 716, Max Score: 2416, Max Tile: 256
Highest Tile Reached (last 100 scores):
128:  45%
256:  3%
512:  

### Save the agent and evaluate 

In [933]:
torch.save(agent.state_dict(), "ppo_2048.pt")

In [ ]:
def load_agent(path, a_size, h_size):
    """Load the saved agent's state dict."""
    agent = Agent(a_size=a_size, h_size=h_size)
    model_weights = torch.load(path, weights_only=True)
    agent.load_state_dict(model_weights)
    agent.eval()
    return agent

In [935]:
path = "ppo_2048.pt"
eval_agent = load_agent(path, a_size, h_size)

In [ ]:
def evaluate_agent(agent):
    """Evluate the agent after fully training the model."""
    game = Game2048(NROW, NCOL)
    state = torch.from_numpy(game.board).float().unsqueeze(0).unsqueeze(0)
    agent.eval()
    with torch.inference_mode():
        while game.game_is_over is False:
            
            # Get action and value
            action, log_prob, entropy, val = agent.get_action_and_value(state)
            action = action.item()

            # Take action with the given policy
            game.take_action(action)

            # Update state and val
            new_state = torch.from_numpy(game.board).float().unsqueeze(0).unsqueeze(0)

            # Update the state
            state = new_state

        # Print game score and board
        print(f"Game Score: {game.total_score}, Max Tile: {2 ** np.max(game.board.flatten())}")
        print(2 ** game.board)
        print()

In [ ]:
# Evaluate the fully trained model
for i in range(1, 6):
    print(f"Evaluation Game {i}")
    evaluate_agent(eval_agent)

Evaluation Game 1
Game Score: 2452, Max Tile: 256
[[  8   4  32   4]
 [  4  16   8   2]
 [ 16  64  32 256]
 [  2   4   2   4]]

Evaluation Game 2
Game Score: 3056, Max Tile: 256
[[  4   2   8   2]
 [ 64  32  16   8]
 [  8 128   4   2]
 [  4 256   2  16]]

Evaluation Game 3
Game Score: 2596, Max Tile: 256
[[  4   2  16   2]
 [  2  64  32   4]
 [  4   8   4  64]
 [  2 256   2   8]]

Evaluation Game 4
Game Score: 4648, Max Tile: 512
[[  4  16   8   2]
 [  8   4 512   4]
 [  4  64   8  32]
 [  2  32   4   2]]

Evaluation Game 5
Game Score: 5004, Max Tile: 512
[[  2   8   4   2]
 [  4 512 128  32]
 [  8  16   4   2]
 [  4   2   8   4]]

